In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
import os

# Dataset Link

# https://drive.google.com/drive/folders/1RdFLeJ66Qq_8-60qwTNTnBSuhWUYacYZ?usp=sharing

BASE = "/content/drive/MyDrive/YawDD-dataset/YawDD dataset"

MIRROR_MALE   = os.path.join(BASE, "Mirror", "Male_mirror Avi Videos")
MIRROR_FEMALE = os.path.join(BASE, "Mirror", "Female_mirror")
DASH_MALE     = os.path.join(BASE, "Dash", "Male")
DASH_FEMALE   = os.path.join(BASE, "Dash", "Female")

for name, path in [("Mirror-Male", MIRROR_MALE), ("Mirror-Female", MIRROR_FEMALE),
                    ("Dash-Male", DASH_MALE), ("Dash-Female", DASH_FEMALE)]:
    if os.path.exists(path):
        files = os.listdir(path)
        print(f"{name}: {len(files)} files, e.g. {files[0] if files else 'EMPTY'}")
    else:
        print(f"{name}: PATH NOT FOUND")

Mirror-Male: 164 files, e.g. 1-MaleNoGlasses-Yawning.avi
Mirror-Female: 156 files, e.g. 1-FemaleNoGlasses-Normal.avi
Dash-Male: 16 files, e.g. 1-MaleGlasses.avi
Dash-Female: 13 files, e.g. 1-FemaleNoGlasses.avi


In [11]:
import os
output_dir = "/content/drive/MyDrive/YawDD-dataset/processed_features"
for f in ["raw_features_all_videos.csv", "train_windows.npy", "val_windows.npy",
          "test_windows.npy", "train_meta.csv", "val_meta.csv", "test_meta.csv"]:
    print(f, os.path.exists(f"{output_dir}/{f}"))

raw_features_all_videos.csv True
train_windows.npy True
val_windows.npy True
test_windows.npy True
train_meta.csv True
val_meta.csv True
test_meta.csv True


In [12]:
"""
MILESTONE 4 — LSTM on FULL dataset, corrected class structure
================================================================
Shiwani — Landmark-Based Fatigue Detection

LSTM already won on the candidate subset, so this script skips GRU/TCN/MLP
entirely and takes LSTM straight to the full dataset after removing the
invalid "Talking_Yawning" class (13/320 = 4.06% < 20% threshold -> dropped).

Paste as new cells after `drive.mount(...)` in a fresh notebook, or after
your existing work. Only needs raw_features_all_videos.csv on Drive.
"""

# ============================================================
# SECTION 0 — SETUP
# ============================================================
import os
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_recall_fscore_support
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

output_dir = "/content/drive/MyDrive/YawDD-dataset/processed_features"
FEATURE_COLS = ["ear", "mar", "pitch", "yaw", "roll"]

raw_df = pd.read_csv(f"{output_dir}/raw_features_all_videos.csv")
clean_df = raw_df.dropna(subset=FEATURE_COLS).copy()
clean_df["label"] = (
    clean_df["label"].str.strip()
    .str.replace("&", "_", regex=False)
    .str.replace("-", "_", regex=False)
)
clean_df["label"] = clean_df["label"].replace({
    "Talking_yawning": "Talking_Yawning", "talking_yawning": "Talking_Yawning",
    "TalkingYawning": "Talking_Yawning", "talkingyawning": "Talking_Yawning",
})








Device: cuda


In [13]:
# ============================================================
# SECTION 1 — DROP "Talking_Yawning" (documented rule: <20% of labeled videos)
# ============================================================
mirror_labeled = clean_df[(clean_df["camera"] == "Mirror") & (clean_df["label"] != "Unknown")]
video_label_counts = mirror_labeled[["video", "label"]].drop_duplicates()["label"].value_counts()
total_videos = video_label_counts.sum()
drop_threshold = 0.20
classes_to_drop = video_label_counts[video_label_counts / total_videos < drop_threshold].index.tolist()

print("Videos per class (before fix):")
for cls, cnt in video_label_counts.items():
    pct = cnt / total_videos * 100
    print(f"  {cls}: {cnt} ({pct:.2f}%)" + (" -> DROPPED" if cls in classes_to_drop else ""))

clean_df = clean_df[~clean_df["label"].isin(classes_to_drop)].copy()
print("\nClasses retained:", sorted(clean_df.loc[clean_df['label'] != 'Unknown', 'label'].unique()))

Videos per class (before fix):
  Normal: 105 (32.81%)
  Yawning: 102 (31.87%)
  Talking: 100 (31.25%)
  Talking_Yawning: 13 (4.06%) -> DROPPED

Classes retained: ['.avi', 'Normal', 'Talking', 'Yawning']


In [14]:

# ============================================================
# SECTION 2 — SUBJECT-LEVEL SPLIT (70/15/15, seed=42, reproducible)
# ============================================================
def get_subject_id(fname):
    m = re.match(r"(\d+)-", fname)
    return m.group(1) if m else "unknown"

clean_df["subject"] = clean_df["video"].apply(get_subject_id)
unique_subjects = clean_df["subject"].unique()
np.random.seed(42)
np.random.shuffle(unique_subjects)
n = len(unique_subjects)
train_subj = set(unique_subjects[:int(0.7 * n)])
val_subj = set(unique_subjects[int(0.7 * n):int(0.85 * n)])
test_subj = set(unique_subjects[int(0.85 * n):])
clean_df["split"] = clean_df["subject"].apply(
    lambda s: "train" if s in train_subj else ("val" if s in val_subj else "test")
)

# Keep only Mirror + labeled (Dash has no action label) -> this IS the full dataset now
clean_df = clean_df[(clean_df["camera"] == "Mirror") & (clean_df["label"] != "Unknown")].copy()
print("\nFull corrected dataset, videos per class:")
print(clean_df[["video", "label"]].drop_duplicates()["label"].value_counts())
print("\nSplit sizes (videos):",
      clean_df[["video", "split"]].drop_duplicates()["split"].value_counts().to_dict())


Full corrected dataset, videos per class:
label
Normal     105
Yawning    102
Talking    100
Name: count, dtype: int64

Split sizes (videos): {'train': 209, 'val': 52, 'test': 46}


In [15]:

# ============================================================
# SECTION 3 — WINDOWING (full dataset, no downsampling)
# ============================================================
label2id = {"Normal": 0, "Talking": 1, "Yawning": 2}
id2label = {v: k for k, v in label2id.items()}
num_classes = len(label2id)

# Fix: a "Yawning" video is only actually yawning for a second or two — the rest
# of that video looks like Normal. Windowing the whole video and stamping every
# window "Yawning" trains the model on mostly-mislabeled examples for that class.
# This relabels a window from "Yawning" -> "Normal" unless its peak MAR (raw,
# not normalized) is clearly elevated relative to THAT SAME VIDEO's own resting
# MAR — self-relative so it's robust to people who naturally have wider/narrower
# mouths. YAWN_TOP_FRAC controls how much of each Yawning video's windows are
# kept as the true yawning moment (0.4 = top 40% by peak MAR kept as Yawning).
YAWN_TOP_FRAC = 0.4

# Tried the same approach for "Talking" using MAR variability (std) as the signal
# for real talking vs. quiet moments in a Talking-labeled video. Measured result:
# it made things WORSE — Talking recall dropped (0.458 -> 0.371) and Yawning F1
# also dropped slightly (0.619 -> 0.587), likely because a single yawn also
# produces high MAR variability, so "top variability" pulled in yawn-like
# excursions rather than isolating real repeated-talking movement, while cutting
# real quieter talking windows. Reverted based on that evidence -- documented
# here as a tested-and-rejected approach, not silently dropped.

def make_windows(df, window_size, stride, feature_cols=FEATURE_COLS):
    df = df.copy()
    train_stats = df[df["split"] == "train"][feature_cols].agg(["mean", "std"])
    for col in feature_cols:
        mean, std = train_stats.loc["mean", col], train_stats.loc["std", col]
        df[f"{col}_norm"] = (df[col] - mean) / (std + 1e-6)
    norm_cols = [f"{c}_norm" for c in feature_cols]

    windows, window_meta = [], []
    for video, group in df.groupby("video"):
        group = group.sort_values("frame_idx").reset_index(drop=True)
        split = group["split"].iloc[0]
        label = group["label"].iloc[0]

        video_windows = []
        for start in range(0, len(group) - window_size + 1, stride):
            w = group.iloc[start:start + window_size]
            if len(w) == window_size:
                peak_mar = w["mar"].max()  # raw MAR, not normalized — Yawning signal
                video_windows.append((start, w[norm_cols].values, peak_mar))

        if label == "Yawning" and len(video_windows) > 0:
            # keep "Yawning" only for the top YAWN_TOP_FRAC windows by peak MAR
            # within THIS video; the rest are relabeled "Normal"
            peaks = np.array([v[2] for v in video_windows])
            cutoff = np.quantile(peaks, 1 - YAWN_TOP_FRAC)
            for start, w_arr, peak_mar in video_windows:
                win_label = "Yawning" if peak_mar >= cutoff else "Normal"
                windows.append(w_arr)
                window_meta.append({"video": video, "split": split, "label": win_label,
                                     "original_video_label": label, "start_frame": start})
        else:
            for start, w_arr, peak_mar in video_windows:
                windows.append(w_arr)
                window_meta.append({"video": video, "split": split, "label": label,
                                     "original_video_label": label, "start_frame": start})

    windows_array = np.array(windows)
    meta_df = pd.DataFrame(window_meta)
    meta_df["label_id"] = meta_df["label"].map(label2id)

    out = {}
    for split_name in ["train", "val", "test"]:
        idx = meta_df[meta_df["split"] == split_name].index.to_numpy()
        out[split_name] = (windows_array[idx], meta_df.loc[idx, "label_id"].values, meta_df.loc[idx])
    return out, train_stats


# NOTE: windowing itself now happens inside the Section 6 sweep (window_size is
# one of the swept hyperparameters), so there's no separate fixed-window build here.

In [16]:

# ============================================================
# SECTION 4 — LOADERS + LSTM MODEL
# ============================================================
def make_loaders(data_dict, batch_size):
    loaders = {}
    for split_name, (X, y, _) in data_dict.items():
        Xt = torch.tensor(X, dtype=torch.float32)
        yt = torch.tensor(y, dtype=torch.long)
        loaders[split_name] = DataLoader(TensorDataset(Xt, yt), batch_size=batch_size,
                                          shuffle=(split_name == "train"))
    return loaders


def compute_class_weights(train_labels, num_classes):
    """Softened inverse-frequency weights (sqrt, not raw) so the minority class
    (Yawning) gets more attention in the loss without dominating it."""
    counts = np.bincount(train_labels, minlength=num_classes)
    total = counts.sum()
    weights = np.sqrt(total / np.maximum(counts, 1))
    return torch.tensor(weights, dtype=torch.float32)


class LSTMModel(nn.Module):
    def __init__(self, input_size=5, hidden_size=64, num_layers=2, dropout=0.3, num_classes=num_classes):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers,
                             batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

In [17]:

# ============================================================
# SECTION 5 — TRAIN / EVAL (with L2 + gradient clipping options)
# ============================================================
def train_model(model, train_loader, val_loader, epochs=25, lr=0.001, weight_decay=0.0,
                 grad_clip=None, class_weights=None):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device) if class_weights is not None else None)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    history = {"train_acc": [], "val_acc": []}
    best_state, best_val_acc = None, 0.0

    for epoch in range(epochs):
        model.train()
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X), y)
            loss.backward()
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
            optimizer.step()

        model.eval()
        val_pred, val_true = [], []
        with torch.no_grad():
            for X, y in val_loader:
                out = model(X.to(device))
                val_pred.extend(out.argmax(1).cpu().numpy())
                val_true.extend(y.numpy())
        val_acc = accuracy_score(val_true, val_pred)
        history["val_acc"].append(val_acc)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.clone().cpu() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history


def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for X, y in loader:
            out = model(X.to(device))
            preds.extend(out.argmax(1).cpu().numpy())
            labels.extend(y.numpy())
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average=None, labels=list(range(num_classes)), zero_division=0)
    cm = confusion_matrix(labels, preds, labels=list(range(num_classes)))
    per_class = {id2label[i]: {"precision": prec[i], "recall": rec[i], "f1": f1[i]} for i in range(num_classes)}
    return {"accuracy": acc, "macro_f1": f1.mean(), "per_class": per_class, "confusion_matrix": cm}, preds


def print_eval(name, result):
    print(f"\n{name}: accuracy={result['accuracy']:.4f}  macro_f1={result['macro_f1']:.4f}")
    for cls, m in result["per_class"].items():
        print(f"  {cls:10s} precision={m['precision']:.3f} recall={m['recall']:.3f} f1={m['f1']:.3f}")
    print("  confusion matrix (rows=true, cols=pred, order=", list(id2label.values()), "):")
    print(result["confusion_matrix"])


In [18]:

# ============================================================
# SECTION 6 — HYPERPARAMETER SWEEP on FULL data: window size, hidden units,
# learning rate, batch size (all four, as required)
# ============================================================
print("\n" + "=" * 60)
print("HYPERPARAMETER SWEEP (full dataset)")
print("=" * 60)

window_sizes = [20, 30, 45]
hidden_sizes = [32, 64, 128]
learning_rates = [0.0005, 0.001]
batch_sizes = [16, 32]
DROPOUT_FOR_SWEEP = 0.3   # dropout itself is handled in the regularization ablation (Section 6b)
CHECK_EPOCHS = 12

windowed_cache = {}
train_stats_cache = {}
for ws in window_sizes:
    windowed_cache[ws], train_stats_cache[ws] = make_windows(clean_df, window_size=ws, stride=ws // 2)
    print(f"window_size={ws}: train={windowed_cache[ws]['train'][0].shape}, "
          f"val={windowed_cache[ws]['val'][0].shape}, test={windowed_cache[ws]['test'][0].shape}")

sweep_results = []
for ws in window_sizes:
    for bs in batch_sizes:
        ws_loaders = make_loaders(windowed_cache[ws], batch_size=bs)
        ws_class_weights = compute_class_weights(windowed_cache[ws]["train"][1], num_classes)
        for hs in hidden_sizes:
            for lr in learning_rates:
                model = LSTMModel(hidden_size=hs, dropout=DROPOUT_FOR_SWEEP)
                model, _ = train_model(model, ws_loaders["train"], ws_loaders["val"], epochs=CHECK_EPOCHS, lr=lr,
                                        class_weights=ws_class_weights)
                result, _ = evaluate(model, ws_loaders["val"])
                sweep_results.append({"window_size": ws, "batch_size": bs, "hidden_size": hs, "lr": lr,
                                       "val_accuracy": result["accuracy"], "val_macro_f1": result["macro_f1"]})
                print(f"ws={ws} bs={bs} hs={hs} lr={lr} -> val_acc={result['accuracy']:.4f} "
                      f"val_macroF1={result['macro_f1']:.4f}")

sweep_df = pd.DataFrame(sweep_results)
sweep_df.to_csv(f"{output_dir}/m4_lstm_fulldata_sweep.csv", index=False)
best_row = sweep_df.loc[sweep_df["val_macro_f1"].idxmax()]
print("\nBest config on full data:\n", best_row)

best_ws = int(best_row["window_size"])
best_bs = int(best_row["batch_size"])
best_hs = int(best_row["hidden_size"])
best_lr = float(best_row["lr"])
loaders = make_loaders(windowed_cache[best_ws], batch_size=best_bs)
class_weights = compute_class_weights(windowed_cache[best_ws]["train"][1], num_classes)
print("Class weights (Normal/Talking/Yawning):", class_weights.tolist())

best_train_stats = train_stats_cache[best_ws]   # raw mean/std, needed for inference on new videos
best_train_stats.to_csv(f"{output_dir}/m4_normalization_stats_ws{best_ws}.csv")
print(f"\nSaved normalization stats for inference (window_size={best_ws}):")
print(best_train_stats)


HYPERPARAMETER SWEEP (full dataset)
window_size=20: train=(14048, 20, 5), val=(3649, 20, 5), test=(3173, 20, 5)
window_size=30: train=(9262, 30, 5), val=(2408, 30, 5), test=(2092, 30, 5)
window_size=45: train=(6204, 45, 5), val=(1614, 45, 5), test=(1400, 45, 5)
ws=20 bs=16 hs=32 lr=0.0005 -> val_acc=0.6514 val_macroF1=0.6317
ws=20 bs=16 hs=32 lr=0.001 -> val_acc=0.6517 val_macroF1=0.6304
ws=20 bs=16 hs=64 lr=0.0005 -> val_acc=0.6402 val_macroF1=0.6410
ws=20 bs=16 hs=64 lr=0.001 -> val_acc=0.6404 val_macroF1=0.6287
ws=20 bs=16 hs=128 lr=0.0005 -> val_acc=0.6457 val_macroF1=0.6153
ws=20 bs=16 hs=128 lr=0.001 -> val_acc=0.6402 val_macroF1=0.5995
ws=20 bs=32 hs=32 lr=0.0005 -> val_acc=0.6303 val_macroF1=0.5931
ws=20 bs=32 hs=32 lr=0.001 -> val_acc=0.6350 val_macroF1=0.6140
ws=20 bs=32 hs=64 lr=0.0005 -> val_acc=0.6383 val_macroF1=0.6094
ws=20 bs=32 hs=64 lr=0.001 -> val_acc=0.6435 val_macroF1=0.6249
ws=20 bs=32 hs=128 lr=0.0005 -> val_acc=0.6388 val_macroF1=0.6005
ws=20 bs=32 hs=128 lr=0.

In [19]:

# ============================================================
# SECTION 6b — REGULARIZATION ABLATION at best config
# (reports the STABILITY IMPACT of each technique individually, not just
# applying all three blindly in the final model)
# ============================================================
print("\n" + "=" * 60)
print("REGULARIZATION ABLATION")
print("=" * 60)

reg_variants = {
    "none":           dict(dropout=0.0, weight_decay=0.0, grad_clip=None),
    "dropout_only":   dict(dropout=0.3, weight_decay=0.0, grad_clip=None),
    "l2_only":        dict(dropout=0.0, weight_decay=1e-4, grad_clip=None),
    "grad_clip_only": dict(dropout=0.0, weight_decay=0.0, grad_clip=1.0),
    "all_combined":   dict(dropout=0.3, weight_decay=1e-4, grad_clip=1.0),
}

ablation_results = {}
for name, cfg in reg_variants.items():
    model = LSTMModel(hidden_size=best_hs, dropout=cfg["dropout"])
    model, history = train_model(model, loaders["train"], loaders["val"], epochs=20, lr=best_lr,
                                  weight_decay=cfg["weight_decay"], grad_clip=cfg["grad_clip"],
                                  class_weights=class_weights)
    stability = float(np.std(history["val_acc"][-5:]))  # lower = more stable
    result, _ = evaluate(model, loaders["test"])
    ablation_results[name] = {"final_val_acc": history["val_acc"][-1], "stability_std": stability,
                               "test_accuracy": result["accuracy"], "test_macro_f1": result["macro_f1"]}
    print(f"{name:16s} final_val_acc={history['val_acc'][-1]:.4f} stability_std={stability:.4f} "
          f"test_acc={result['accuracy']:.4f} test_macroF1={result['macro_f1']:.4f}")

ablation_df = pd.DataFrame(ablation_results).T
ablation_df.to_csv(f"{output_dir}/m4_lstm_regularization_ablation.csv")
best_reg_name = ablation_df["test_macro_f1"].astype(float).idxmax()
best_reg_cfg = reg_variants[best_reg_name]
print(f"\nBest regularization combo: {best_reg_name} -> {best_reg_cfg}")


REGULARIZATION ABLATION
none             final_val_acc=0.6388 stability_std=0.0250 test_acc=0.6221 test_macroF1=0.5983
dropout_only     final_val_acc=0.6549 stability_std=0.0193 test_acc=0.6071 test_macroF1=0.5852
l2_only          final_val_acc=0.6283 stability_std=0.0170 test_acc=0.6400 test_macroF1=0.6237
grad_clip_only   final_val_acc=0.6388 stability_std=0.0113 test_acc=0.6264 test_macroF1=0.6014
all_combined     final_val_acc=0.6506 stability_std=0.0084 test_acc=0.6257 test_macroF1=0.6070

Best regularization combo: l2_only -> {'dropout': 0.0, 'weight_decay': 0.0001, 'grad_clip': None}


In [20]:

# ============================================================
# SECTION 7 — FINAL MODEL: best config + regularization (L2 + grad clip), full epochs
# ============================================================
print("\n" + "=" * 60)
print("FINAL LSTM (full dataset, corrected classes)")
print("=" * 60)

final_model = LSTMModel(hidden_size=best_hs, dropout=best_reg_cfg["dropout"])
final_model, history = train_model(
    final_model, loaders["train"], loaders["val"], epochs=25, lr=best_lr,
    weight_decay=best_reg_cfg["weight_decay"], grad_clip=best_reg_cfg["grad_clip"],
    class_weights=class_weights
)
stability = float(np.std(history["val_acc"][-5:]))
print(f"Val-accuracy stability (std, last 5 epochs): {stability:.4f}")

test_result, test_preds = evaluate(final_model, loaders["test"])
print_eval("AFTER FIX (test set)", test_result)

torch.save(final_model.state_dict(), f"{output_dir}/m4_lstm_full_final.pt")


FINAL LSTM (full dataset, corrected classes)
Val-accuracy stability (std, last 5 epochs): 0.0146

AFTER FIX (test set): accuracy=0.6386  macro_f1=0.6191
  Normal     precision=0.625 recall=0.784 f1=0.695
  Talking    precision=0.654 recall=0.510 f1=0.573
  Yawning    precision=0.669 recall=0.525 f1=0.589
  confusion matrix (rows=true, cols=pred, order= ['Normal', 'Talking', 'Yawning'] ):
[[508 113  27]
 [277 303  14]
 [ 28  47  83]]


In [21]:

# ============================================================
# SECTION 7b — "BEFORE FIX" BASELINE: reload the OLD 4-class windows already
# saved from Milestone 3 (untouched by this script) so the before/after
# comparison your task requires is a real, reproduced result — not just a
# remembered number.
# ============================================================
print("\n" + "=" * 60)
print("BEFORE FIX: LSTM on old 4-class structure (incl. Talking_Yawning)")
print("=" * 60)

old_label2id = {"Normal": 0, "Talking": 1, "Yawning": 2, "Talking_Yawning": 3}
old_id2label = {v: k for k, v in old_label2id.items()}

X_train_old = np.load(f"{output_dir}/train_windows.npy")
X_val_old = np.load(f"{output_dir}/val_windows.npy")
X_test_old = np.load(f"{output_dir}/test_windows.npy")
train_meta_old = pd.read_csv(f"{output_dir}/train_meta.csv")
val_meta_old = pd.read_csv(f"{output_dir}/val_meta.csv")
test_meta_old = pd.read_csv(f"{output_dir}/test_meta.csv")

old_loaders = {}
for split_name, X, meta in [("train", X_train_old, train_meta_old),
                             ("val", X_val_old, val_meta_old),
                             ("test", X_test_old, test_meta_old)]:
    Xt = torch.tensor(X, dtype=torch.float32)
    yt = torch.tensor(meta["label_id"].values, dtype=torch.long)
    old_loaders[split_name] = DataLoader(TensorDataset(Xt, yt), batch_size=32,
                                          shuffle=(split_name == "train"))

before_model = LSTMModel(hidden_size=32, dropout=0.3, num_classes=len(old_label2id))
before_model, _ = train_model(before_model, old_loaders["train"], old_loaders["val"], epochs=15, lr=0.0005)

before_model.eval()
before_preds, before_labels = [], []
with torch.no_grad():
    for X, y in old_loaders["test"]:
        out = before_model(X.to(device))
        before_preds.extend(out.argmax(1).cpu().numpy())
        before_labels.extend(y.numpy())
before_acc = accuracy_score(before_labels, before_preds)
before_prec, before_rec, before_f1, _ = precision_recall_fscore_support(
    before_labels, before_preds, average=None, labels=list(range(len(old_label2id))), zero_division=0)
before_cm = confusion_matrix(before_labels, before_preds, labels=list(range(len(old_label2id))))

print(f"\nBEFORE FIX: accuracy={before_acc:.4f}  macro_f1={before_f1.mean():.4f}")
for i, cls in old_id2label.items():
    print(f"  {cls:16s} precision={before_prec[i]:.3f} recall={before_rec[i]:.3f} f1={before_f1[i]:.3f}")
print("  confusion matrix (order=", list(old_id2label.values()), "):")
print(before_cm)

torch.save(before_model.state_dict(), f"{output_dir}/m4_lstm_before_fix.pt")


BEFORE FIX: LSTM on old 4-class structure (incl. Talking_Yawning)

BEFORE FIX: accuracy=0.4219  macro_f1=0.3898
  Normal           precision=0.465 recall=0.782 f1=0.583
  Talking          precision=0.379 recall=0.353 f1=0.366
  Yawning          precision=0.390 recall=0.457 f1=0.421
  Talking_Yawning  precision=0.455 recall=0.120 f1=0.190
  confusion matrix (order= ['Normal', 'Talking', 'Yawning', 'Talking_Yawning'] ):
[[93 22  4  0]
 [59 47 26  1]
 [16 30 53 17]
 [32 25 53 15]]


In [22]:
# ============================================================
# SECTION 7c — BEFORE vs AFTER COMPARISON TABLE
# ============================================================
comparison = pd.DataFrame({
    "accuracy": [before_acc, test_result["accuracy"]],
    "macro_f1": [before_f1.mean(), test_result["macro_f1"]],
}, index=["before_fix (4-class, incl. Talking_Yawning)", "after_fix (3-class, corrected)"])
print("\n" + "=" * 60)
print("BEFORE vs AFTER SUMMARY")
print("=" * 60)
print(comparison)
comparison.to_csv(f"{output_dir}/m4_before_after_comparison.csv")


BEFORE vs AFTER SUMMARY
                                             accuracy  macro_f1
before_fix (4-class, incl. Talking_Yawning)  0.421907  0.389835
after_fix (3-class, corrected)               0.638571  0.619129


In [23]:

# ============================================================
# SECTION 8 — RELIABILITY GATE before deriving fatigue states
# ============================================================
MIN_YAWN_F1 = 0.50
yawn_f1 = test_result["per_class"]["Yawning"]["f1"]
print(f"\nYawning-class F1: {yawn_f1:.3f} (threshold: {MIN_YAWN_F1})")

FATIGUE_THRESHOLD = 0.15
def actions_to_fatigue_state(action_sequence, threshold=FATIGUE_THRESHOLD):
    action_sequence = list(action_sequence)
    T = len(action_sequence)
    if T == 0:
        return "Unknown"
    p_yawn = sum(1 for a in action_sequence if a == "Yawning") / T
    if p_yawn == 0:
        return "Alert"
    elif p_yawn < threshold:
        return "Mild Fatigue"
    return "Drowsy"

if yawn_f1 < MIN_YAWN_F1:
    print("DECISION: Yawning F1 below threshold -> reporting raw action labels "
          "(Normal/Talking/Yawning) only; fatigue-state derivation not applied.")
else:
    print("DECISION: Yawning F1 meets threshold -> deriving fatigue states.")
    test_meta = windowed_cache[best_ws]["test"][2].reset_index(drop=True)
    test_meta["pred_action"] = [id2label[p] for p in test_preds]
    for video, group in list(test_meta.groupby("video"))[:5]:
        state = actions_to_fatigue_state(group["pred_action"].tolist())
        print(f"  {video}: true_video_label={group['original_video_label'].iloc[0]:10s} "
              f"preds={group['pred_action'].tolist()} -> {state}")


Yawning-class F1: 0.589 (threshold: 0.5)
DECISION: Yawning F1 meets threshold -> deriving fatigue states.
  11-FemaleNoGlasses-Normal.avi: true_video_label=Normal     preds=['Normal', 'Normal', 'Normal', 'Talking', 'Talking', 'Normal', 'Talking', 'Talking', 'Talking', 'Normal', 'Normal', 'Talking', 'Talking', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal'] -> Alert
  11-FemaleNoGlasses-Talking.avi: true_video_label=Talking    preds=['Talking', 'Talking', 'Talking', 'Talking', 'Normal', 'Talking', 'Talking', 'Talking', 'Normal', 'Normal', 'Talking', 'Talking', 'Normal', 'Normal', 'Talking', 'Talking', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Talking', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Talking', 'Normal', 'Normal', 'Talking', 'Talking', 'Talking', 'Normal', 'Talking', 'Talking', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Normal', 'Talking'] -> Alert
  11-FemaleNoGlasses-Yawning.avi

In [24]:
# ============================================================
# SECTION 9 — ARTIFACTS
# ============================================================
print("\nSaved artifacts:")
for f in ["m4_lstm_fulldata_sweep.csv", "m4_lstm_full_final.pt", f"m4_normalization_stats_ws{best_ws}.csv"]:
    path = f"{output_dir}/{f}"
    print(" -", path, "(exists)" if os.path.exists(path) else "(MISSING)")


Saved artifacts:
 - /content/drive/MyDrive/YawDD-dataset/processed_features/m4_lstm_fulldata_sweep.csv (exists)
 - /content/drive/MyDrive/YawDD-dataset/processed_features/m4_lstm_full_final.pt (exists)
 - /content/drive/MyDrive/YawDD-dataset/processed_features/m4_normalization_stats_ws45.csv (exists)


In [33]:

# ============================================================
# SECTION 10 — INFERENCE ON A BRAND-NEW VIDEO
# ============================================================
# NOTE: this model only works on VIDEO (a sequence of frames), not a single
# image — it needs to see EAR/MAR/pose CHANGE over best_ws consecutive frames
# to tell Normal/Talking/Yawning apart. A single photo has no temporal
# information for it to use.
!pip install mediapipe -q

import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
import math
from collections import Counter

LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
MOUTH     = [61, 291, 39, 181, 0, 17]

def _euclid(p1, p2):
    return math.dist(p1, p2)

def _compute_ear(lm, eye_idx, w, h):
    pts = [(lm[i].x * w, lm[i].y * h) for i in eye_idx]
    v1, v2 = _euclid(pts[1], pts[5]), _euclid(pts[2], pts[4])
    horiz = _euclid(pts[0], pts[3])
    return (v1 + v2) / (2.0 * horiz + 1e-6)

def _compute_mar(lm, mouth_idx, w, h):
    pts = [(lm[i].x * w, lm[i].y * h) for i in mouth_idx]
    vert = _euclid(pts[2], pts[4])
    horiz = _euclid(pts[0], pts[1])
    return vert / (horiz + 1e-6)

def _compute_pose(matrix):
    rmat = np.array(matrix)[:3, :3]
    sy = math.sqrt(rmat[0, 0] ** 2 + rmat[1, 0] ** 2)
    pitch = math.degrees(math.atan2(-rmat[2, 0], sy))
    yaw = math.degrees(math.atan2(rmat[1, 0], rmat[0, 0]))
    roll = math.degrees(math.atan2(rmat[2, 1], rmat[2, 2]))
    return pitch, yaw, roll

base_options = mp_python.BaseOptions(
    model_asset_path='/content/drive/MyDrive/YawDD-dataset/face_landmarker.task'
)
_detector = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=base_options, num_faces=1,
    output_facial_transformation_matrixes=True, running_mode=vision.RunningMode.IMAGE
))

# raw normalization stats from the actual training split (fixes the earlier bug
# of normalizing with stats taken from already-normalized window data)
INFER_MEAN = np.array([best_train_stats.loc["mean", c] for c in FEATURE_COLS])
INFER_STD = np.array([best_train_stats.loc["std", c] for c in FEATURE_COLS])

def predict_video(video_path, model, window_size=best_ws, stride=best_ws // 2):
    if not os.path.exists(video_path):
        return None, f"File not found: {video_path}"

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, f"OpenCV could not open this video (bad path or unsupported codec): {video_path}"

    raw_features = []
    n_frames_read = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        n_frames_read += 1
        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = _detector.detect(mp_image)
        if result.face_landmarks:
            lm = result.face_landmarks[0]
            ear = (_compute_ear(lm, LEFT_EYE, w, h) + _compute_ear(lm, RIGHT_EYE, w, h)) / 2
            mar = _compute_mar(lm, MOUTH, w, h)
            pitch, yaw, roll = _compute_pose(result.facial_transformation_matrixes[0])
            raw_features.append([ear, mar, pitch, yaw, roll])
        else:
            raw_features.append([np.nan] * 5)
    cap.release()

    if n_frames_read == 0:
        return None, (f"OpenCV opened the file but read 0 frames — likely an unsupported "
                       f"codec in this Colab environment. Try converting to .mp4 first, e.g. "
                       f"with: !ffmpeg -i '{video_path}' -y /content/converted.mp4  "
                       f"then pass '/content/converted.mp4' instead.")

    raw_features = np.array(raw_features)
    valid = ~np.isnan(raw_features).any(axis=1)
    clean_features = raw_features[valid]
    if len(clean_features) < window_size:
        return None, "Video too short (or too many failed-detection frames) for one full window."

    normalized = (clean_features - INFER_MEAN) / (INFER_STD + 1e-6)
    windows = [normalized[s:s + window_size] for s in range(0, len(normalized) - window_size + 1, stride)]
    windows = np.array(windows)

    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(windows, dtype=torch.float32).to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

    per_window = [{"window": i, "predicted_action": id2label[p],
                   "confidence": float(probs[i, p])} for i, p in enumerate(preds)]

    result = {"num_windows": len(windows), "per_window_predictions": per_window}
    if yawn_f1 >= MIN_YAWN_F1:
        result["fatigue_state"] = actions_to_fatigue_state([id2label[p] for p in preds])
    return result, None

# --- Example usage ---
# result, error = predict_video("/content/drive/MyDrive/YawDD-dataset/YawDD dataset/Mirror/Male_mirror Avi Videos/some_video.avi", final_model)
# if error:
#     print(error)
# else:
#     for pw in result["per_window_predictions"]:
#         print(pw)
#     if "fatigue_state" in result:
#         print("Overall fatigue state for this video:", result["fatigue_state"])


def summarize_prediction(result, true_label=None):
    """Quick at-a-glance summary of a predict_video() result: a timeline of
    predicted actions + confidence, and the derived fatigue state (the actual
    headline output). Majority-vote video-level action is NOT reported as a
    final answer -- it's a separate diagnostic only shown when true_label is
    given (i.e. when evaluating raw window-level accuracy against known
    ground truth on a test-set video). See report Section 2 for why: the
    fatigue-state rule looks only at yawning-window proportion, independent
    of whichever action wins the majority vote -- a video can correctly
    derive "Drowsy" from a real sustained yawn even while "Talking" wins the
    majority vote, and reporting majority vote as if it were the answer would
    make that correct result look wrong."""
    preds = [pw["predicted_action"] for pw in result["per_window_predictions"]]
    confs = [pw["confidence"] for pw in result["per_window_predictions"]]

    print(f"Windows: {len(preds)}")
    for pw in result["per_window_predictions"]:
        bar = "#" * int(pw["confidence"] * 20)
        print(f"  win {pw['window']:3d}: {pw['predicted_action']:8s} "
              f"conf={pw['confidence']:.2f} {bar}")

    print(f"\nAverage confidence: {np.mean(confs):.3f}  "
          f"(min={np.min(confs):.3f}, max={np.max(confs):.3f})")

    # consistency check: how often does the prediction change between
    # consecutive windows? Low = stable/trustworthy, high = noisy/uncertain
    flips = sum(1 for i in range(1, len(preds)) if preds[i] != preds[i - 1])
    print(f"Prediction changes between consecutive windows: {flips}/{len(preds) - 1}")

    if true_label is not None:
        # diagnostic only -- for comparing raw window-level classification
        # against ground truth during testing, not a deployment output
        majority = Counter(preds).most_common(1)[0][0]
        match = "MATCH" if majority == true_label else "MISMATCH"
        print(f"\n[diagnostic] Majority-vote action: {majority} | True label: {true_label} -> {match}")

    if "fatigue_state" in result:
        print(f"\nFatigue state (headline output): {result['fatigue_state']}")
    else:
        print("\nFatigue state: not derived (Yawning-class F1 below reliability threshold)")


In [25]:

# ============================================================
# SECTION 10 — INFERENCE ON A BRAND-NEW VIDEO
# ============================================================
# NOTE: this model only works on VIDEO (a sequence of frames), not a single
# image — it needs to see EAR/MAR/pose CHANGE over best_ws consecutive frames
# to tell Normal/Talking/Yawning apart. A single photo has no temporal
# information for it to use.
!pip install mediapipe -q

import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision
import math
from collections import Counter

LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
MOUTH     = [61, 291, 39, 181, 0, 17]

def _euclid(p1, p2):
    return math.dist(p1, p2)

def _compute_ear(lm, eye_idx, w, h):
    pts = [(lm[i].x * w, lm[i].y * h) for i in eye_idx]
    v1, v2 = _euclid(pts[1], pts[5]), _euclid(pts[2], pts[4])
    horiz = _euclid(pts[0], pts[3])
    return (v1 + v2) / (2.0 * horiz + 1e-6)

def _compute_mar(lm, mouth_idx, w, h):
    pts = [(lm[i].x * w, lm[i].y * h) for i in mouth_idx]
    vert = _euclid(pts[2], pts[4])
    horiz = _euclid(pts[0], pts[1])
    return vert / (horiz + 1e-6)

def _compute_pose(matrix):
    rmat = np.array(matrix)[:3, :3]
    sy = math.sqrt(rmat[0, 0] ** 2 + rmat[1, 0] ** 2)
    pitch = math.degrees(math.atan2(-rmat[2, 0], sy))
    yaw = math.degrees(math.atan2(rmat[1, 0], rmat[0, 0]))
    roll = math.degrees(math.atan2(rmat[2, 1], rmat[2, 2]))
    return pitch, yaw, roll

base_options = mp_python.BaseOptions(
    model_asset_path='/content/drive/MyDrive/YawDD-dataset/face_landmarker.task'
)
_detector = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=base_options, num_faces=1,
    output_facial_transformation_matrixes=True, running_mode=vision.RunningMode.IMAGE
))

# raw normalization stats from the actual training split (fixes the earlier bug
# of normalizing with stats taken from already-normalized window data)
INFER_MEAN = np.array([best_train_stats.loc["mean", c] for c in FEATURE_COLS])
INFER_STD = np.array([best_train_stats.loc["std", c] for c in FEATURE_COLS])

def predict_video(video_path, model, window_size=best_ws, stride=best_ws // 2):
    if not os.path.exists(video_path):
        return None, f"File not found: {video_path}"

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None, f"OpenCV could not open this video (bad path or unsupported codec): {video_path}"

    raw_features = []
    n_frames_read = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        n_frames_read += 1
        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = _detector.detect(mp_image)
        if result.face_landmarks:
            lm = result.face_landmarks[0]
            ear = (_compute_ear(lm, LEFT_EYE, w, h) + _compute_ear(lm, RIGHT_EYE, w, h)) / 2
            mar = _compute_mar(lm, MOUTH, w, h)
            pitch, yaw, roll = _compute_pose(result.facial_transformation_matrixes[0])
            raw_features.append([ear, mar, pitch, yaw, roll])
        else:
            raw_features.append([np.nan] * 5)
    cap.release()

    if n_frames_read == 0:
        return None, (f"OpenCV opened the file but read 0 frames — likely an unsupported "
                       f"codec in this Colab environment. Try converting to .mp4 first, e.g. "
                       f"with: !ffmpeg -i '{video_path}' -y /content/converted.mp4  "
                       f"then pass '/content/converted.mp4' instead.")

    raw_features = np.array(raw_features)
    valid = ~np.isnan(raw_features).any(axis=1)
    clean_features = raw_features[valid]
    if len(clean_features) < window_size:
        return None, "Video too short (or too many failed-detection frames) for one full window."

    normalized = (clean_features - INFER_MEAN) / (INFER_STD + 1e-6)
    windows = [normalized[s:s + window_size] for s in range(0, len(normalized) - window_size + 1, stride)]
    windows = np.array(windows)

    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(windows, dtype=torch.float32).to(device))
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

    per_window = [{"window": i, "predicted_action": id2label[p],
                   "confidence": float(probs[i, p])} for i, p in enumerate(preds)]

    result = {"num_windows": len(windows), "per_window_predictions": per_window}
    if yawn_f1 >= MIN_YAWN_F1:
        result["fatigue_state"] = actions_to_fatigue_state([id2label[p] for p in preds])
    return result, None


def summarize_prediction(result, true_label=None):
    """Quick at-a-glance summary of a predict_video() result: a timeline of
    predicted actions + confidence, plus a majority-vote video-level label.
    If true_label is given (only possible for test-set videos with a known
    ground truth), also reports whether the majority vote matched it."""
    preds = [pw["predicted_action"] for pw in result["per_window_predictions"]]
    confs = [pw["confidence"] for pw in result["per_window_predictions"]]

    print(f"Windows: {len(preds)}")
    for pw in result["per_window_predictions"]:
        bar = "#" * int(pw["confidence"] * 20)
        print(f"  win {pw['window']:3d}: {pw['predicted_action']:8s} "
              f"conf={pw['confidence']:.2f} {bar}")

    majority = Counter(preds).most_common(1)[0][0]
    print(f"\nMajority-vote video label: {majority}")
    print(f"Average confidence: {np.mean(confs):.3f}  "
          f"(min={np.min(confs):.3f}, max={np.max(confs):.3f})")

    # consistency check: how often does the prediction change between
    # consecutive windows? Low = stable/trustworthy, high = noisy/uncertain
    flips = sum(1 for i in range(1, len(preds)) if preds[i] != preds[i - 1])
    print(f"Prediction changes between consecutive windows: {flips}/{len(preds) - 1}")

    if true_label is not None:
        match = "MATCH" if majority == true_label else "MISMATCH"
        print(f"\nTrue label: {true_label} -> {match}")

    if "fatigue_state" in result:
        print(f"\nDerived fatigue state: {result['fatigue_state']}")



In [27]:
 #Example: run + summarize on a KNOWN test-set video (majority vote vs true label)
test_meta_final = windowed_cache[best_ws]["test"][2]
sample_video = test_meta_final["video"].iloc[0]
true_lbl = test_meta_final[test_meta_final["video"] == sample_video]["label"].iloc[0]
video_path = f"/content/drive/MyDrive/YawDD-dataset/YawDD dataset/Mirror/.../{sample_video}"
result, error = predict_video(video_path, final_model)
if not error:
    summarize_prediction(result, true_label=true_lbl)

In [34]:
video_path = "/content/47-MaleNoGlasses-Yawning.avi"   # wherever you saved it

result, error = predict_video(video_path, final_model)
if error:
    print(error)
else:
    summarize_prediction(result)

Windows: 23
  win   0: Normal   conf=0.50 ##########
  win   1: Normal   conf=0.62 ############
  win   2: Normal   conf=0.74 ##############
  win   3: Normal   conf=0.54 ##########
  win   4: Normal   conf=0.45 #########
  win   5: Talking  conf=0.71 ##############
  win   6: Talking  conf=0.58 ###########
  win   7: Normal   conf=0.79 ###############
  win   8: Normal   conf=0.51 ##########
  win   9: Normal   conf=0.60 ###########
  win  10: Talking  conf=0.61 ############
  win  11: Talking  conf=0.71 ##############
  win  12: Yawning  conf=0.96 ###################
  win  13: Yawning  conf=0.98 ###################
  win  14: Yawning  conf=0.58 ###########
  win  15: Talking  conf=0.84 ################
  win  16: Talking  conf=0.83 ################
  win  17: Talking  conf=0.56 ###########
  win  18: Normal   conf=0.72 ##############
  win  19: Normal   conf=0.81 ################
  win  20: Normal   conf=0.70 ##############
  win  21: Normal   conf=0.77 ###############
  win  22: Ta

In [35]:
video_path = "/content/47-MaleNoGlasses-Talking.avi"   # wherever you saved it

result, error = predict_video(video_path, final_model)
if error:
    print(error)
else:
    summarize_prediction(result)   # no true_label — you don't have one for this video

Windows: 47
  win   0: Talking  conf=0.67 #############
  win   1: Talking  conf=0.59 ###########
  win   2: Talking  conf=0.72 ##############
  win   3: Talking  conf=0.74 ##############
  win   4: Talking  conf=0.76 ###############
  win   5: Talking  conf=0.65 ############
  win   6: Talking  conf=0.76 ###############
  win   7: Talking  conf=0.79 ###############
  win   8: Talking  conf=0.82 ################
  win   9: Talking  conf=0.84 ################
  win  10: Talking  conf=0.84 ################
  win  11: Talking  conf=0.75 ###############
  win  12: Talking  conf=0.90 #################
  win  13: Talking  conf=0.89 #################
  win  14: Talking  conf=0.89 #################
  win  15: Talking  conf=0.86 #################
  win  16: Talking  conf=0.84 ################
  win  17: Talking  conf=0.75 ##############
  win  18: Talking  conf=0.72 ##############
  win  19: Talking  conf=0.51 ##########
  win  20: Talking  conf=0.78 ###############
  win  21: Talking  conf=0.8

In [36]:
video_path = "/content/46-MaleGlasses-Yawning.avi"   # wherever you saved it

result, error = predict_video(video_path, final_model)
if error:
    print(error)
else:
    summarize_prediction(result)   # no true_label — you don't have one for this video

Windows: 23
  win   0: Talking  conf=0.86 #################
  win   1: Talking  conf=0.70 #############
  win   2: Talking  conf=0.70 ##############
  win   3: Talking  conf=0.43 ########
  win   4: Talking  conf=0.46 #########
  win   5: Talking  conf=0.59 ###########
  win   6: Talking  conf=0.86 #################
  win   7: Talking  conf=0.65 ############
  win   8: Talking  conf=0.68 #############
  win   9: Talking  conf=0.42 ########
  win  10: Yawning  conf=0.74 ##############
  win  11: Yawning  conf=0.93 ##################
  win  12: Yawning  conf=0.98 ###################
  win  13: Yawning  conf=0.99 ###################
  win  14: Yawning  conf=0.99 ###################
  win  15: Yawning  conf=0.96 ###################
  win  16: Talking  conf=0.63 ############
  win  17: Talking  conf=0.50 ##########
  win  18: Yawning  conf=0.62 ############
  win  19: Talking  conf=0.67 #############
  win  20: Talking  conf=0.82 ################
  win  21: Talking  conf=0.62 ############
 

In [37]:
video_path = "/content/47-MaleNoGlasses-Normal.avi"   # wherever you saved it

result, error = predict_video(video_path, final_model)
if error:
    print(error)
else:
    summarize_prediction(result)   # no true_label — you don't have one for this video

Windows: 28
  win   0: Talking  conf=0.41 ########
  win   1: Yawning  conf=0.79 ###############
  win   2: Normal   conf=0.44 ########
  win   3: Talking  conf=0.57 ###########
  win   4: Talking  conf=0.85 #################
  win   5: Talking  conf=0.62 ############
  win   6: Normal   conf=0.77 ###############
  win   7: Normal   conf=0.72 ##############
  win   8: Normal   conf=0.74 ##############
  win   9: Normal   conf=0.73 ##############
  win  10: Normal   conf=0.74 ##############
  win  11: Normal   conf=0.79 ###############
  win  12: Yawning  conf=0.39 #######
  win  13: Talking  conf=0.44 ########
  win  14: Talking  conf=0.69 #############
  win  15: Normal   conf=0.59 ###########
  win  16: Talking  conf=0.53 ##########
  win  17: Normal   conf=0.52 ##########
  win  18: Talking  conf=0.69 #############
  win  19: Talking  conf=0.60 ############
  win  20: Talking  conf=0.45 #########
  win  21: Normal   conf=0.78 ###############
  win  22: Normal   conf=0.80 ###########

In [26]:
# ============================================================
# SECTION 11 — STREAMING PREDICTION for team integration
# (Kushagra's orchestration pipeline: rolling buffer, updates every new frame)
# ============================================================
# NOTE for the team: unlike Shubham's per-frame confidence (a single frame is
# enough evidence for texting/phone detection), this model needs a WINDOW of
# frames to work at all -- a single frame carries no information about mouth
# or head movement over time. So this module's natural update unit is "once
# per full window" (best_ws frames), not once per single frame. Confidence is
# still reported every time, same shape as the other modules -- it just
# refreshes once per window-length instead of every frame.

# Keep a running history of recent fatigue_states so a short buffer of ONE
# window's worth of predictions can still be voted, exactly like the
# actions_to_fatigue_state() rule expects a sequence, not a single value.
_recent_action_history = []
FATIGUE_HISTORY_LENGTH = 10   # how many recent window-predictions to vote over

def predict_from_buffer(frame_buffer, model, window_size=best_ws):
    """
    frame_buffer: a list of the most recent RAW camera frames (numpy arrays,
                   BGR, as read by cv2.VideoCapture / a live camera). Must be
                   at least `window_size` frames long -- Kushagra's orchestrator
                   is responsible for maintaining this rolling buffer and
                   calling this function once it's full.

    Returns a dict ready to hand to the Risk Fusion Engine:
        predicted_action : "Normal" / "Talking" / "Yawning"   (this window only)
        confidence        : float 0-1, softmax probability of predicted_action
        fatigue_state     : "Alert" / "Mild Fatigue" / "Drowsy" / None
                            (None if the Yawning-F1 reliability gate hasn't
                            been met -- see Section 8 -- in which case only
                            report predicted_action, per that documented rule)
        yawn_proportion   : float 0-1, the same p_Y used internally by the
                            voting rule -- hand this to Ravina's fusion engine
                            directly if she wants a continuous risk signal
                            instead of just the 3-bucket fatigue_state label.
    """
    if len(frame_buffer) < window_size:
        return None, f"Buffer not full yet: {len(frame_buffer)}/{window_size} frames"

    recent_frames = frame_buffer[-window_size:]
    raw_features = []
    for frame in recent_frames:
        h, w, _ = frame.shape
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        result = _detector.detect(mp_image)
        if result.face_landmarks:
            lm = result.face_landmarks[0]
            ear = (_compute_ear(lm, LEFT_EYE, w, h) + _compute_ear(lm, RIGHT_EYE, w, h)) / 2
            mar = _compute_mar(lm, MOUTH, w, h)
            pitch, yaw, roll = _compute_pose(result.facial_transformation_matrixes[0])
            raw_features.append([ear, mar, pitch, yaw, roll])
        else:
            raw_features.append([np.nan] * 5)

    raw_features = np.array(raw_features)
    if np.isnan(raw_features).any():
        # a face-detection dropout inside this window -- report as a failure
        # mode Kushagra's orchestrator should handle (per his own spec:
        # "a module producing no detection")
        return None, "Face not detected in one or more frames of this window"

    normalized = (raw_features - INFER_MEAN) / (INFER_STD + 1e-6)
    window_tensor = torch.tensor(normalized[np.newaxis, :, :], dtype=torch.float32).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(window_tensor)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
        pred_id = int(probs.argmax())

    predicted_action = id2label[pred_id]
    confidence = float(probs[pred_id])

    global _recent_action_history
    _recent_action_history.append(predicted_action)
    _recent_action_history = _recent_action_history[-FATIGUE_HISTORY_LENGTH:]

    yawn_proportion = _recent_action_history.count("Yawning") / len(_recent_action_history)

    output = {
        "predicted_action": predicted_action,
        "confidence": confidence,
        "yawn_proportion": yawn_proportion,
        "fatigue_state": None,
    }
    if yawn_f1 >= MIN_YAWN_F1:
        output["fatigue_state"] = actions_to_fatigue_state(_recent_action_history)

    return output, None


In [ ]:
# --- Example usage (simulating a live stream from a recorded file) ---
cap = cv2.VideoCapture("/content/drive/.../some_video.avi")
buffer = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    buffer.append(frame)
    if len(buffer) >= best_ws:
        output, error = predict_from_buffer(buffer, final_model)
        if error:
            print(error)
        else:
            print(output)
cap.release()